# E2 — Transformer: Multi-Head Self-Attention và tokenization

**CO5085 · Học sâu và ứng dụng trong thị giác máy tính · HK261** — tập dữ liệu: Fashion-MNIST (cùng split với E1).

Notebook này tự chứa toàn bộ code của E2 và chạy được từ đầu đến cuối (khoảng 15–20 phút trên GPU T4).

| Yêu cầu của đề | Phần trong notebook |
|---|---|
| 1. MSA tự viết (Linear, matmul, softmax, ghép head) | `MultiHeadSelfAttentionScratch` |
| 2. MSA bằng PyTorch để đối chiếu | `MultiHeadSelfAttentionTorch` (`nn.MultiheadAttention`) |
| 3. Đưa MSA vào bộ phân loại ảnh (encoder nông + head) | `ImageTransformer` |
| 4. Ít nhất hai cách lấy token | patch 4×4, patch 7×7, theo hàng, CNN-stem |
| 5. So sánh tự viết vs PyTorch; so sánh các tokenizer | Mục *Kết quả* |

## Thiết lập
Chạy lần lượt từ trên xuống. Trên Colab: *Runtime → Change runtime type → T4 GPU*.

In [ ]:
import csv
import json
import math
import os
import random
import time
from dataclasses import asdict, dataclass, field

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from IPython.display import display
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch", torch.__version__, "| thiết bị:",
      torch.cuda.get_device_name(0) if device.type == "cuda" else "CPU (sẽ chậm, nên bật GPU)")

In [ ]:
# ---- hằng số và hàm vẽ/đọc kết quả (không cần PyTorch)
from matplotlib.ticker import MaxNLocator

CLASSES = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
           "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]
SHORT = ["T-shirt", "Trouser", "Pullover", "Dress", "Coat", "Sandal", "Shirt", "Sneaker", "Bag", "Boot"]
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
INK2, MUTED, GRID = "#52514e", "#8a8984", "#e4e3df"

plt.rcParams.update({
    "figure.dpi": 110, "font.size": 9, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6, "axes.axisbelow": True,
    "legend.frameon": False, "lines.linewidth": 1.8, "savefig.bbox": "tight",
})


def load_run(run_dir):
    """Đọc lại một lần chạy đã lưu: summary.json, history.csv, logits trên tập test."""
    with open(os.path.join(run_dir, "summary.json"), encoding="utf-8") as f:
        summary = json.load(f)
    run = {"summary": summary, "history": pd.read_csv(os.path.join(run_dir, "history.csv"))}
    npz = os.path.join(run_dir, "test_outputs.npz")
    if os.path.isfile(npz):
        d = np.load(npz)
        run["logits"], run["targets"] = d["logits"], d["targets"]
        run["preds"] = run["logits"].argmax(1)
    return run


def summary_row(label, run):
    s = run["summary"]
    return {"Mô hình": label, "Số tham số": s["params"],
            "Train acc (%)": round(100 * s["final_train_acc"], 2),
            "Val acc tốt nhất (%)": round(100 * s["best_val_acc"], 2), "Epoch tốt nhất": s["best_epoch"],
            "Test acc (%)": round(100 * s["test_acc"], 2), "Test loss": round(s["test_loss"], 4),
            "Giây/epoch": round(s["mean_epoch_time"], 2), "Suy luận test (s)": round(s["test_inference_time"], 3)}


def plot_curves(runs, labels, colors, styles=None, save_as=None):
    """Loss và accuracy theo epoch. Nét chấm: train; nét liền/gạch: validation."""
    styles = styles or {}
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.8))
    for key, run in runs.items():
        h, c, ls = run["history"], colors[key], styles.get(key, "-")
        ax1.plot(h["epoch"], h["train_loss"], color=c, ls=":", lw=1.3)
        ax1.plot(h["epoch"], h["val_loss"], color=c, ls=ls)
        ax2.plot(h["epoch"], 100 * h["train_acc"], color=c, ls=":", lw=1.3)
        ax2.plot(h["epoch"], 100 * h["val_acc"], color=c, ls=ls, label=labels[key])
    ax1.set(xlabel="epoch", ylabel="cross-entropy loss", title="Loss")
    ax2.set(xlabel="epoch", ylabel="accuracy (%)", title="Accuracy")
    ax2.legend(fontsize=8)
    for ax in (ax1, ax2):
        ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    fig.text(0.5, -0.02, "nét chấm: train · nét liền/gạch: validation", ha="center", color=INK2)
    fig.tight_layout()
    if save_as:
        fig.savefig(save_as)
    plt.show()


def confusion(targets, preds, k=10):
    cm = np.zeros((k, k), dtype=int)
    np.add.at(cm, (targets, preds), 1)
    return cm


def softmax_np(z):
    z = z - z.max(1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(1, keepdims=True)

In [ ]:
EXERCISE = "E2"
SEED = 42
DATA_ROOT = "data"
RESULTS = "results/E2"
FIGURES = "figures/E2"
BATCH_SIZE = 128
QUICK = False          # True: mỗi mô hình chỉ 1 epoch, để chạy thử toàn bộ notebook thật nhanh
                       # (chạy thử xong nhớ xóa thư mục results/ trước khi chạy thật)
EPOCHS = 1 if QUICK else 20
LR = 1e-3
WEIGHT_DECAY = 0.05
WARMUP_EPOCHS = 1.0
EMBED_DIM, DEPTH, HEADS, MLP_RATIO, DROPOUT = 96, 4, 4, 2.0, 0.1
# Mỗi lần chạy là "tokenizer:cài_đặt_MSA". Thêm "patch4s2:scratch" để thử patch chồng nhau (169 token, chậm hơn).
RUNS = ["patch4:scratch", "patch4:torch", "row:scratch", "row:torch", "cnn:scratch", "patch7:scratch"]
TOK_LABEL = {"patch4": "Patch 4×4", "patch7": "Patch 7×7", "patch4s2": "Patch 4×4 chồng",
             "row": "Theo hàng", "cnn": "CNN-stem"}
IMPL_LABEL = {"scratch": "tự viết", "torch": "PyTorch"}
os.makedirs(RESULTS, exist_ok=True)
os.makedirs(FIGURES, exist_ok=True)

### Seed, đo thời gian, lưu file

In [ ]:
def set_seed(seed: int = 42, deterministic: bool = True) -> None:
    """Cố định mọi nguồn ngẫu nhiên để kết quả có thể tái lập."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


def get_device() -> torch.device:
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def count_params(model: torch.nn.Module, trainable_only: bool = True) -> int:
    """Tổng số phần tử của các tham số (p.numel())."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad or not trainable_only)


def sync(device: torch.device) -> None:
    """Chờ GPU chạy xong trước khi đọc đồng hồ, nếu không thời gian đo sẽ sai."""
    if device.type == "cuda":
        torch.cuda.synchronize()


class Timer:
    """Dùng với `with Timer(device) as t: ...` rồi đọc t.elapsed (giây)."""

    def __init__(self, device: torch.device):
        self.device = device
        self.elapsed = 0.0

    def __enter__(self):
        sync(self.device)
        self._start = time.perf_counter()
        return self

    def __exit__(self, *exc):
        sync(self.device)
        self.elapsed = time.perf_counter() - self._start
        return False


def save_json(obj, path: str) -> None:
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)


def load_json(path: str):
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def environment_info() -> dict:
    """Ghi lại môi trường chạy để đưa vào báo cáo."""
    info = {
        "torch": torch.__version__,
        "cuda_available": torch.cuda.is_available(),
        "device_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
    }
    try:
        import torchvision

        info["torchvision"] = torchvision.__version__
    except ImportError:
        pass
    return info

## Dữ liệu: Fashion-MNIST

- 60 000 ảnh train gốc được hoán vị bằng seed cố định: **54 000 train**, **6 000 validation**.
  10 000 ảnh test gốc chỉ dùng cho kết quả cuối.
- Ảnh đưa về $[0,1]$ rồi chuẩn hóa bằng mean/std tính **chỉ trên phần train**. Không tăng cường dữ liệu.
- Toàn bộ ảnh được giữ sẵn dưới dạng tensor trong bộ nhớ (28×28 nên rất nhẹ), mỗi epoch không phải đọc lại từ đĩa.
- Cả ba bài dùng đúng cùng split này (cùng seed), nên kết quả so sánh được với nhau.

In [ ]:
CLASSES = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot",
]
NUM_CLASSES = 10
IMAGE_SHAPE = (1, 28, 28)


@dataclass
class DataBundle:
    train: DataLoader
    val: DataLoader
    test: DataLoader
    mean: float
    std: float
    sizes: dict


def _load_raw(root: str):
    from torchvision import datasets

    train = datasets.FashionMNIST(root, train=True, download=True)
    test = datasets.FashionMNIST(root, train=False, download=True)
    # .data: uint8 (N, 28, 28), .targets: int64 (N,)
    return train.data, train.targets, test.data, test.targets


def split_indices(n_total: int, val_size: int, seed: int):
    """Trả về (train_idx, val_idx) từ một hoán vị cố định."""
    g = torch.Generator().manual_seed(seed)
    perm = torch.randperm(n_total, generator=g)
    return perm[val_size:], perm[:val_size]


def get_data(root: str = "data", batch_size: int = 128, val_size: int = 6000,
             split_seed: int = 42, loader_seed: int = 0, eval_batch_size: int = 1000) -> DataBundle:
    x_tr_all, y_tr_all, x_te, y_te = _load_raw(root)
    train_idx, val_idx = split_indices(len(x_tr_all), val_size, split_seed)

    x_tr = x_tr_all[train_idx].float().div_(255.0)
    x_va = x_tr_all[val_idx].float().div_(255.0)
    x_te = x_te.float().div_(255.0)

    mean, std = x_tr.mean().item(), x_tr.std().item()

    def prep(x):  # (N, 28, 28) -> (N, 1, 28, 28), chuẩn hóa theo thống kê của tập train
        return ((x - mean) / std).unsqueeze(1)

    ds_tr = TensorDataset(prep(x_tr), y_tr_all[train_idx])
    ds_va = TensorDataset(prep(x_va), y_tr_all[val_idx])
    ds_te = TensorDataset(prep(x_te), y_te)

    # Generator riêng cho DataLoader: cùng loader_seed -> cùng thứ tự batch giữa các lần chạy.
    g = torch.Generator().manual_seed(loader_seed)
    pin = torch.cuda.is_available()
    train_loader = DataLoader(ds_tr, batch_size=batch_size, shuffle=True, generator=g, pin_memory=pin)
    val_loader = DataLoader(ds_va, batch_size=eval_batch_size, shuffle=False, pin_memory=pin)
    test_loader = DataLoader(ds_te, batch_size=eval_batch_size, shuffle=False, pin_memory=pin)

    sizes = {"train": len(ds_tr), "val": len(ds_va), "test": len(ds_te)}
    return DataBundle(train_loader, val_loader, test_loader, mean, std, sizes)


def black_value(mean: float, std: float) -> float:
    """Giá trị của một pixel đen (0) sau khi chuẩn hóa; dùng để che một phần ảnh ở E3."""
    return (0.0 - mean) / std

In [ ]:
data = get_data(DATA_ROOT, batch_size=BATCH_SIZE, loader_seed=SEED)
print("Kích thước:", data.sizes, "| mean = %.4f, std = %.4f" % (data.mean, data.std))

# Ảnh test gốc (uint8, chưa chuẩn hóa) để hiển thị
TEST_IMAGES = datasets.FashionMNIST(DATA_ROOT, train=False, download=True).data.numpy()
TEST_LABELS = datasets.FashionMNIST(DATA_ROOT, train=False, download=True).targets.numpy()

fig, axes = plt.subplots(2, 10, figsize=(12, 2.8))
for c in range(10):
    idx = np.where(TEST_LABELS == c)[0][:2]
    for r in range(2):
        axes[r, c].imshow(TEST_IMAGES[idx[r]], cmap="gray_r")
        axes[r, c].axis("off")
    axes[0, c].set_title(CLASSES[c], fontsize=8)
plt.suptitle("Hai ảnh mẫu của mỗi lớp")
plt.show()

## Vòng lặp huấn luyện tự viết

Mỗi bước đi đúng trình tự **forward → loss → `backward()` → `optimizer.step()`**, không dùng `trainer.fit()`.

- **Loss:** `nn.CrossEntropyLoss` nhận logits (đã gồm log-softmax), nên mô hình không đặt softmax ở cuối `forward`.
- **Optimizer:** AdamW, không áp weight decay cho bias, tham số chuẩn hóa, token CLS và positional embedding.
- **Learning rate:** đổi theo từng bước, warmup tuyến tính (nếu có) rồi giảm theo cosine về 0.
- **Chọn mô hình:** sau mỗi epoch đánh giá trên validation, giữ checkpoint có val accuracy cao nhất;
  tập test chỉ được đánh giá một lần bằng checkpoint đó.
- **Lưu kết quả:** mỗi lần chạy ghi `history.csv`, `summary.json`, `best.pt`, `test_outputs.npz`
  (cùng định dạng với script trong repo, nên `scripts/make_report_assets.py` dùng lại được).

In [ ]:
@dataclass
class TrainConfig:
    epochs: int = 15
    lr: float = 1e-3
    weight_decay: float = 1e-4
    warmup_epochs: float = 0.0      # số epoch tăng LR tuyến tính từ 0 trước khi giảm theo cosine
    grad_clip: float = 0.0          # 0 = không cắt gradient
    label_smoothing: float = 0.0
    seed: int = 42
    extra: dict = field(default_factory=dict)  # thông tin riêng của từng bài (kiến trúc, tokenizer...)


def make_optimizer(model: nn.Module, cfg: TrainConfig) -> torch.optim.Optimizer:
    """AdamW; không áp weight decay lên bias, LayerNorm/BatchNorm, CLS token và positional embedding."""
    decay, no_decay = [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if p.ndim <= 1 or name.endswith(".bias") or "pos_embed" in name or "cls_token" in name:
            no_decay.append(p)
        else:
            decay.append(p)
    groups = [
        {"params": decay, "weight_decay": cfg.weight_decay},
        {"params": no_decay, "weight_decay": 0.0},
    ]
    groups = [g for g in groups if g["params"]]
    return torch.optim.AdamW(groups, lr=cfg.lr)


def make_scheduler(optimizer, cfg: TrainConfig, steps_per_epoch: int):
    """LR theo từng bước: warmup tuyến tính rồi giảm theo cosine về 0."""
    total = cfg.epochs * steps_per_epoch
    warmup = int(cfg.warmup_epochs * steps_per_epoch)

    def lr_lambda(step: int) -> float:
        if warmup > 0 and step < warmup:
            return (step + 1) / warmup
        progress = (step - warmup) / max(1, total - warmup)
        return 0.5 * (1.0 + math.cos(math.pi * min(1.0, progress)))

    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


def train_one_epoch(model, loader, optimizer, scheduler, criterion, device, grad_clip=0.0):
    model.train()
    total_loss, correct, n = 0.0, 0, 0
    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        logits = model(x)                    # 1. forward: (B, 10) logits, CHƯA qua softmax
        loss = criterion(logits, y)          # 2. loss: CrossEntropyLoss = log-softmax + NLL

        optimizer.zero_grad(set_to_none=True)
        loss.backward()                      # 3. backward: tính gradient
        if grad_clip > 0:
            nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        optimizer.step()                     # 4. cập nhật trọng số
        scheduler.step()                     # LR thay đổi theo từng bước

        bs = y.size(0)
        total_loss += loss.item() * bs
        correct += (logits.argmax(dim=1) == y).sum().item()
        n += bs
    return total_loss / n, correct / n


@torch.no_grad()
def evaluate(model, loader, criterion, device, return_outputs=False):
    model.eval()
    total_loss, correct, n = 0.0, 0, 0
    all_logits, all_targets = [], []
    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        logits = model(x)
        total_loss += criterion(logits, y).item() * y.size(0)
        correct += (logits.argmax(dim=1) == y).sum().item()
        n += y.size(0)
        if return_outputs:
            all_logits.append(logits.float().cpu())
            all_targets.append(y.cpu())
    if return_outputs:
        return total_loss / n, correct / n, torch.cat(all_logits), torch.cat(all_targets)
    return total_loss / n, correct / n


def fit(model: nn.Module, data, cfg: TrainConfig, device, out_dir: str, name: str) -> dict:
    """Huấn luyện, chọn checkpoint tốt nhất theo val accuracy, đánh giá trên test, lưu log.

    Ghi ra out_dir:
        history.csv   - loss/acc train & val, LR, thời gian từng epoch
        best.pt       - trọng số có val accuracy cao nhất
        test_outputs.npz - logits và nhãn trên tập test (để vẽ confusion matrix, tìm mẫu sai)
        summary.json  - tóm tắt số liệu + cấu hình + môi trường
    """
    os.makedirs(out_dir, exist_ok=True)
    model.to(device)
    criterion = nn.CrossEntropyLoss(label_smoothing=cfg.label_smoothing)
    optimizer = make_optimizer(model, cfg)
    scheduler = make_scheduler(optimizer, cfg, len(data.train))

    n_params = count_params(model)
    print(f"[{name}] params = {n_params:,} | device = {device}")

    history = []
    best_val_acc, best_epoch = -1.0, -1
    ckpt_path = os.path.join(out_dir, "best.pt")
    for epoch in range(1, cfg.epochs + 1):
        lr_now = optimizer.param_groups[0]["lr"]
        with Timer(device) as t:
            tr_loss, tr_acc = train_one_epoch(model, data.train, optimizer, scheduler,
                                              criterion, device, cfg.grad_clip)
        va_loss, va_acc = evaluate(model, data.val, criterion, device)
        history.append({
            "epoch": epoch, "train_loss": tr_loss, "train_acc": tr_acc,
            "val_loss": va_loss, "val_acc": va_acc, "lr": lr_now, "epoch_time": t.elapsed,
        })
        if va_acc > best_val_acc:
            best_val_acc, best_epoch = va_acc, epoch
            torch.save(model.state_dict(), ckpt_path)
        print(f"[{name}] epoch {epoch:2d}/{cfg.epochs} | train loss {tr_loss:.4f} acc {tr_acc:.4f} "
              f"| val loss {va_loss:.4f} acc {va_acc:.4f} | {t.elapsed:.1f}s")

    with open(os.path.join(out_dir, "history.csv"), "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(history[0].keys()))
        writer.writeheader()
        writer.writerows(history)

    # Đánh giá trên test bằng checkpoint tốt nhất theo val (không chọn theo test).
    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    te_loss, te_acc, logits, targets = evaluate(model, data.test, criterion, device, return_outputs=True)
    np.savez_compressed(os.path.join(out_dir, "test_outputs.npz"),
                        logits=logits.numpy().astype(np.float32), targets=targets.numpy())

    # Thời gian suy luận trên toàn bộ tập test (sau 1 lượt chạy làm nóng).
    evaluate(model, data.test, criterion, device)
    with Timer(device) as t_inf:
        evaluate(model, data.test, criterion, device)

    epoch_times = [h["epoch_time"] for h in history]
    summary = {
        "name": name,
        "params": n_params,
        "best_epoch": best_epoch,
        "best_val_acc": best_val_acc,
        "test_acc": te_acc,
        "test_loss": te_loss,
        "final_train_acc": history[-1]["train_acc"],
        "final_train_loss": history[-1]["train_loss"],
        "mean_epoch_time": float(np.mean(epoch_times[1:] if len(epoch_times) > 1 else epoch_times)),
        "total_train_time": float(np.sum(epoch_times)),
        "test_inference_time": t_inf.elapsed,
        "data_sizes": data.sizes,
        "norm_mean": data.mean,
        "norm_std": data.std,
        "config": asdict(cfg),
        "env": environment_info(),
    }
    save_json(summary, os.path.join(out_dir, "summary.json"))
    print(f"[{name}] best val acc {best_val_acc:.4f} (epoch {best_epoch}) | test acc {te_acc:.4f}")
    return summary

## 1. Multi-Head Self-Attention tự viết

Với $X\in\mathbb{R}^{N\times D}$, $h$ head, $d_k = D/h$:

$$Q = XW_Q,\; K = XW_K,\; V = XW_V,\qquad
A_i = \mathrm{softmax}\!\Big(\frac{Q_iK_i^\top}{\sqrt{d_k}}\Big),\qquad
\mathrm{MSA}(X) = \mathrm{concat}(A_1V_1,\dots,A_hV_h)\,W_O$$

Theo dõi shape: `(B, N, D)` → chiếu Q, K, V → tách head `(B, h, N, d_k)` → điểm `(B, h, N, N)` → softmax theo
chiều cuối → nhân V → ghép head về `(B, N, D)` → `out_proj`.
Bản tự viết chỉ dùng `nn.Linear`, `torch.matmul`, `torch.softmax` và đổi shape; bản đối chiếu bọc `nn.MultiheadAttention`.

In [ ]:
class MultiHeadSelfAttentionScratch(nn.Module):
    def __init__(self, embed_dim: int, num_heads: int, attn_dropout: float = 0.0, bias: bool = True):
        super().__init__()
        assert embed_dim % num_heads == 0, "embed_dim phải chia hết cho num_heads"
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.scale = 1.0 / math.sqrt(self.head_dim)

        self.q_proj = nn.Linear(embed_dim, embed_dim, bias=bias)
        self.k_proj = nn.Linear(embed_dim, embed_dim, bias=bias)
        self.v_proj = nn.Linear(embed_dim, embed_dim, bias=bias)
        self.out_proj = nn.Linear(embed_dim, embed_dim, bias=bias)
        self.attn_drop = nn.Dropout(attn_dropout)
        self._reset_parameters()

    @torch.no_grad()
    def _reset_parameters(self):
        # Khởi tạo cùng kiểu với nn.MultiheadAttention: xavier_uniform trên ma trận ghép
        # [W_Q; W_K; W_V] kích thước (3D, D), bias = 0; out_proj giữ khởi tạo mặc định của
        # nn.Linear nhưng bias = 0. Nhờ vậy hai cài đặt bắt đầu từ cùng một phân phối trọng số.
        D = self.embed_dim
        w = torch.empty(3 * D, D)
        nn.init.xavier_uniform_(w)
        self.q_proj.weight.copy_(w[:D])
        self.k_proj.weight.copy_(w[D:2 * D])
        self.v_proj.weight.copy_(w[2 * D:])
        for lin in (self.q_proj, self.k_proj, self.v_proj, self.out_proj):
            if lin.bias is not None:
                lin.bias.zero_()

    def _split_heads(self, t: torch.Tensor) -> torch.Tensor:
        B, N, _ = t.shape
        # (B, N, D) -> (B, N, h, d_k) -> (B, h, N, d_k)
        return t.view(B, N, self.num_heads, self.head_dim).transpose(1, 2)

    def forward(self, x: torch.Tensor, return_attn: bool = False):
        B, N, D = x.shape
        q = self._split_heads(self.q_proj(x))                       # (B, h, N, d_k)
        k = self._split_heads(self.k_proj(x))                       # (B, h, N, d_k)
        v = self._split_heads(self.v_proj(x))                       # (B, h, N, d_k)

        scores = torch.matmul(q, k.transpose(-2, -1)) * self.scale  # (B, h, N, N)
        attn = torch.softmax(scores, dim=-1)                        # mỗi hàng cộng lại bằng 1
        attn_dropped = self.attn_drop(attn)

        out = torch.matmul(attn_dropped, v)                         # (B, h, N, d_k)
        out = out.transpose(1, 2).reshape(B, N, D)                  # ghép head: (B, N, D)
        out = self.out_proj(out)
        return (out, attn) if return_attn else out

    @torch.no_grad()
    def load_from_torch(self, mha: nn.MultiheadAttention) -> None:
        """Chép trọng số từ nn.MultiheadAttention để kiểm chứng hai cài đặt cho cùng kết quả.

        mha.in_proj_weight có shape (3D, D) = [W_Q; W_K; W_V] xếp chồng theo hàng.
        """
        D = self.embed_dim
        W = mha.in_proj_weight
        self.q_proj.weight.copy_(W[:D])
        self.k_proj.weight.copy_(W[D:2 * D])
        self.v_proj.weight.copy_(W[2 * D:])
        if mha.in_proj_bias is not None:
            b = mha.in_proj_bias
            self.q_proj.bias.copy_(b[:D])
            self.k_proj.bias.copy_(b[D:2 * D])
            self.v_proj.bias.copy_(b[2 * D:])
        self.out_proj.weight.copy_(mha.out_proj.weight)
        if mha.out_proj.bias is not None:
            self.out_proj.bias.copy_(mha.out_proj.bias)


class MultiHeadSelfAttentionTorch(nn.Module):
    """Bản đối chiếu dùng nn.MultiheadAttention (batch_first=True để nhận (B, N, D))."""

    def __init__(self, embed_dim: int, num_heads: int, attn_dropout: float = 0.0, bias: bool = True):
        super().__init__()
        self.mha = nn.MultiheadAttention(embed_dim, num_heads, dropout=attn_dropout,
                                         bias=bias, batch_first=True)

    def forward(self, x: torch.Tensor, return_attn: bool = False):
        # need_weights=False cho phép PyTorch dùng kernel scaled_dot_product_attention nhanh hơn.
        out, attn = self.mha(x, x, x, need_weights=return_attn, average_attn_weights=False)
        return (out, attn) if return_attn else out


def build_msa(impl: str, embed_dim: int, num_heads: int, attn_dropout: float = 0.0) -> nn.Module:
    if impl == "scratch":
        return MultiHeadSelfAttentionScratch(embed_dim, num_heads, attn_dropout)
    if impl == "torch":
        return MultiHeadSelfAttentionTorch(embed_dim, num_heads, attn_dropout)
    raise ValueError(f"impl không hợp lệ: {impl}")

### Kiểm chứng: MSA tự viết == `nn.MultiheadAttention`

Tạo `nn.MultiheadAttention` ngẫu nhiên, chép trọng số sang bản tự viết (`in_proj_weight` là $[W_Q; W_K; W_V]$
xếp chồng), đưa cùng đầu vào qua cả hai rồi so **đầu ra**, **ma trận attention từng head** và **gradient theo đầu vào**,
với cả float64 và float32.

In [ ]:
def compare_msa(D, h, N, dtype, batch=8):
    ref = nn.MultiheadAttention(D, h, batch_first=True).to(device=device, dtype=dtype).eval()
    mine = MultiHeadSelfAttentionScratch(D, h).to(device=device, dtype=dtype).eval()
    mine.load_from_torch(ref)
    x1 = torch.randn(batch, N, D, device=device, dtype=dtype, requires_grad=True)
    x2 = x1.detach().clone().requires_grad_(True)
    out_ref, attn_ref = ref(x1, x1, x1, need_weights=True, average_attn_weights=False)
    out_mine, attn_mine = mine(x2, return_attn=True)
    g = torch.randn_like(out_ref)       # cùng một gradient ở đầu ra cho cả hai
    out_ref.backward(g)
    out_mine.backward(g)
    return {"embed_dim": D, "num_heads": h, "num_tokens": N, "dtype": str(dtype).replace("torch.", ""),
            "max_abs_diff_output": (out_ref - out_mine).abs().max().item(),
            "max_abs_diff_attention": (attn_ref - attn_mine).abs().max().item(),
            "max_abs_diff_input_grad": (x1.grad - x2.grad).abs().max().item(),
            "attention_rows_sum_to_one": bool(torch.allclose(attn_mine.sum(-1), torch.ones_like(attn_mine.sum(-1)))),
            "params_scratch": sum(p.numel() for p in mine.parameters()),
            "params_torch": sum(p.numel() for p in ref.parameters())}


set_seed(0)
check_rows = []
for dtype, tol in [(torch.float64, 1e-10), (torch.float32, 1e-4)]:
    for D, h, N in [(32, 4, 10), (96, 4, 50), (96, 8, 29), (64, 1, 17)]:
        r = compare_msa(D, h, N, dtype)
        worst = max(r["max_abs_diff_output"], r["max_abs_diff_attention"], r["max_abs_diff_input_grad"])
        r["tolerance"] = tol
        r["passed"] = bool(worst < tol and r["attention_rows_sum_to_one"] and r["params_scratch"] == r["params_torch"])
        check_rows.append(r)
save_json({"device": str(device), "results": check_rows}, os.path.join(RESULTS, "msa_check.json"))
assert all(r["passed"] for r in check_rows), "MSA tự viết không khớp nn.MultiheadAttention"
print("MSA tự viết khớp nn.MultiheadAttention ở mọi cấu hình.")

In [ ]:
with open(os.path.join(RESULTS, "msa_check.json"), encoding="utf-8") as f:
    check = pd.DataFrame(json.load(f)["results"])
display(check[["embed_dim", "num_heads", "num_tokens", "dtype", "max_abs_diff_output",
               "max_abs_diff_attention", "max_abs_diff_input_grad", "params_scratch", "params_torch", "passed"]])

### Chi phí một lớp MSA theo số token $N$

Forward + backward một lớp MSA với batch 128, $D=96$, $h=4$. $N$ = 16 (patch 7×7), 49 (patch 4×4),
169 (patch chồng), 196 (patch 2×2), 784 (mỗi pixel một token). Ma trận attention có $B\cdot h\cdot N^2$ phần tử.

In [ ]:
def bench_ms(fn, x, iters):
    for _ in range(3):                       # làm nóng
        fn(x).sum().backward()
    with Timer(device) as t:
        for _ in range(iters):
            fn(x).sum().backward()
    return t.elapsed / iters * 1000.0


def peak_mem_mb(fn, x):
    if device.type != "cuda":
        return None
    torch.cuda.reset_peak_memory_stats()
    fn(x).sum().backward()
    torch.cuda.synchronize()
    return torch.cuda.max_memory_allocated() / 2 ** 20


set_seed(0)
B_BENCH, ITERS = 128, (20 if device.type == "cuda" else 2)
ref_mha = nn.MultiheadAttention(EMBED_DIM, HEADS, batch_first=True).to(device)
mine_msa = MultiHeadSelfAttentionScratch(EMBED_DIM, HEADS).to(device)
mine_msa.load_from_torch(ref_mha)
impls = {"scratch": lambda x: mine_msa(x), "torch": lambda x: ref_mha(x, x, x, need_weights=False)[0]}

bench_rows = []
for n in [16, 49, 169, 196, 784]:
    x = torch.randn(B_BENCH, n, EMBED_DIM, device=device, requires_grad=True)
    row = {"num_tokens": n, "batch": B_BENCH, "attn_matrix_mb": B_BENCH * HEADS * n * n * 4 / 2 ** 20}
    for impl, fn in impls.items():
        try:
            row[f"{impl}_ms"] = bench_ms(fn, x, ITERS)
            row[f"{impl}_peak_mem_mb"] = peak_mem_mb(fn, x)
        except torch.cuda.OutOfMemoryError:
            row[f"{impl}_ms"] = row[f"{impl}_peak_mem_mb"] = None
            torch.cuda.empty_cache()
    bench_rows.append(row)
    print(row)
save_json({"device": str(device), "embed_dim": EMBED_DIM, "heads": HEADS, "iters": ITERS, "results": bench_rows},
          os.path.join(RESULTS, "msa_bench.json"))

In [ ]:
with open(os.path.join(RESULTS, "msa_bench.json"), encoding="utf-8") as f:
    bench = pd.DataFrame(json.load(f)["results"])
display(bench.round(2))
fig, ax = plt.subplots(figsize=(5.5, 3.6))
for impl, c in [("scratch", SERIES[0]), ("torch", SERIES[1])]:
    ok = bench[f"{impl}_ms"].notna()
    ax.plot(bench["num_tokens"][ok], bench[f"{impl}_ms"][ok], marker="o", ms=5, color=c, label=f"MSA {IMPL_LABEL[impl]}")
ax.set(xscale="log", yscale="log", xlabel="số token N", ylabel="ms / lượt (forward + backward)",
       title="Chi phí một lớp MSA theo số token")
ax.set_xticks([16, 49, 196, 784], ["16", "49", "196", "784"])
ax.minorticks_off()
ax.legend()
fig.savefig(os.path.join(FIGURES, "e2_bench.png"))
plt.show()

## 2. Các cách lấy token

| Tokenizer | Mỗi token là | Số token |
|---|---|---|
| `patch4` | ô 4×4 không chồng nhau, chiếu tuyến tính 16 → D | 49 |
| `patch7` | ô 7×7 không chồng nhau, chiếu tuyến tính 49 → D | 16 |
| `patch4s2` | ô 4×4 với stride 2 (chồng nhau) | 169 |
| `row` | một hàng 28 pixel, chiếu tuyến tính 28 → D | 28 |
| `cnn` | một vị trí trên feature map 7×7 của CNN 2 tầng + conv 1×1 về D kênh | 49 |

Patch embedding được viết bằng `nn.Conv2d(kernel_size = stride = p)`. Ô dưới chứng minh nó **tương đương**
cắt ảnh thành ô p×p, flatten rồi nhân cùng một ma trận chiếu.

In [ ]:
H = W = 28


class PatchTokenizer(nn.Module):
    """Patch embedding.

    nn.Conv2d với kernel_size = stride = p tương đương hoàn toàn với:
    cắt ảnh thành các ô p x p không chồng nhau, flatten mỗi ô (C*p*p giá trị),
    rồi nhân với cùng một ma trận chiếu tuyến tính (C*p*p -> D).
    Viết bằng Conv2d chỉ vì gọn và nhanh hơn.
    """

    def __init__(self, embed_dim: int, patch_size: int = 4, stride: int | None = None, in_ch: int = 1):
        super().__init__()
        stride = stride or patch_size
        self.proj = nn.Conv2d(in_ch, embed_dim, kernel_size=patch_size, stride=stride)
        side = (H - patch_size) // stride + 1
        self.num_tokens = side * side
        self.grid = (side, side)

    def forward(self, x):                       # (B, C, 28, 28)
        x = self.proj(x)                        # (B, D, side, side)
        return x.flatten(2).transpose(1, 2)     # (B, side*side, D)


class RowTokenizer(nn.Module):
    """Mỗi hàng ảnh (C*28 giá trị) là một token, chiếu tuyến tính lên D chiều."""

    def __init__(self, embed_dim: int, in_ch: int = 1):
        super().__init__()
        self.proj = nn.Linear(in_ch * W, embed_dim)
        self.num_tokens = H

    def forward(self, x):                                   # (B, C, 28, 28)
        B, C, Hh, Ww = x.shape
        rows = x.permute(0, 2, 1, 3).reshape(B, Hh, C * Ww)  # (B, 28, C*28)
        return self.proj(rows)                              # (B, 28, D)


class CNNStemTokenizer(nn.Module):
    """CNN nhỏ: 28x28 -> 14x14 -> 7x7, sau đó conv 1x1 đưa số kênh về D.
    Mỗi vị trí trên lưới 7x7 là một token (49 token), mang thông tin của một vùng ảnh
    lớn hơn một patch 4x4 nhờ receptive field của các tầng conv.
    """

    def __init__(self, embed_dim: int, channels=(32, 64), in_ch: int = 1):
        super().__init__()
        layers, prev = [], in_ch
        for c in channels:
            layers += [
                nn.Conv2d(prev, c, kernel_size=3, padding=1, bias=False),
                nn.BatchNorm2d(c),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(2),
            ]
            prev = c
        layers.append(nn.Conv2d(prev, embed_dim, kernel_size=1))
        self.stem = nn.Sequential(*layers)
        side = H // (2 ** len(channels))
        self.num_tokens = side * side
        self.grid = (side, side)

    def forward(self, x):
        x = self.stem(x)                        # (B, D, 7, 7)
        return x.flatten(2).transpose(1, 2)     # (B, 49, D)


TOKENIZERS = {
    "patch4": lambda d: PatchTokenizer(d, patch_size=4),
    "patch7": lambda d: PatchTokenizer(d, patch_size=7),
    "patch4s2": lambda d: PatchTokenizer(d, patch_size=4, stride=2),   # patch chồng nhau: 169 token
    "row": lambda d: RowTokenizer(d),
    "cnn": lambda d: CNNStemTokenizer(d),
}


def build_tokenizer(name: str, embed_dim: int) -> nn.Module:
    if name not in TOKENIZERS:
        raise ValueError(f"tokenizer không hợp lệ: {name}; chọn trong {list(TOKENIZERS)}")
    return TOKENIZERS[name](embed_dim)


def patchify(x: torch.Tensor, p: int) -> torch.Tensor:
    """Cắt ảnh thành patch bằng thao tác tensor thuần (dùng trong test để chứng minh
    PatchTokenizer == patchify + Linear). (B, C, H, W) -> (B, (H/p)*(W/p), C*p*p)."""
    B, C, Hh, Ww = x.shape
    x = x.unfold(2, p, p).unfold(3, p, p)              # (B, C, H/p, W/p, p, p)
    x = x.permute(0, 2, 3, 1, 4, 5)                     # (B, H/p, W/p, C, p, p)
    return x.reshape(B, (Hh // p) * (Ww // p), C * p * p)

In [ ]:
x_demo = torch.randn(2, 1, 28, 28)
for name in ["patch4", "patch7", "patch4s2", "row", "cnn"]:
    tok = build_tokenizer(name, EMBED_DIM)
    print(f"{name:9s} -> token {tuple(tok(x_demo).shape)}  (num_tokens = {tok.num_tokens})")

# Conv2d(k = s = p) == patchify + Linear
tok = PatchTokenizer(32, patch_size=4)
lin = nn.Linear(16, 32)
with torch.no_grad():
    lin.weight.copy_(tok.proj.weight.reshape(32, -1))
    lin.bias.copy_(tok.proj.bias)
print("Conv2d patch embedding == patchify + Linear:",
      torch.allclose(tok(x_demo), lin(patchify(x_demo, 4)), atol=1e-5))

## 3. Bộ phân loại: Transformer encoder nông

ảnh → tokenizer → thêm token **CLS** → cộng **positional embedding** (học được) → 4 khối encoder → LayerNorm → Linear trên CLS.

Mỗi khối dạng *pre-norm*: $X \leftarrow X + \mathrm{MSA}(\mathrm{LN}(X))$, $X \leftarrow X + \mathrm{FFN}(\mathrm{LN}(X))$,
với FFN = Linear–GELU–Linear. Khối encoder được tự lắp (không dùng `nn.TransformerEncoderLayer`) để bản tự viết
và bản PyTorch **chỉ khác nhau đúng ở phần MSA**.

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, dim: int, hidden: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, hidden), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden, dim), nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class EncoderBlock(nn.Module):
    def __init__(self, dim: int, num_heads: int, mlp_ratio: float, dropout: float,
                 attn_impl: str, attn_dropout: float = 0.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = build_msa(attn_impl, dim, num_heads, attn_dropout)
        self.drop = nn.Dropout(dropout)
        self.norm2 = nn.LayerNorm(dim)
        self.ffn = FeedForward(dim, int(dim * mlp_ratio), dropout)

    def forward(self, x):
        x = x + self.drop(self.attn(self.norm1(x)))
        x = x + self.ffn(self.norm2(x))
        return x


class ImageTransformer(nn.Module):
    def __init__(self, tokenizer: str = "patch4", attn_impl: str = "scratch", embed_dim: int = 96,
                 depth: int = 4, num_heads: int = 4, mlp_ratio: float = 2.0, dropout: float = 0.1,
                 attn_dropout: float = 0.0, num_classes: int = 10):
        super().__init__()
        self.tokenizer = build_tokenizer(tokenizer, embed_dim)
        n = self.tokenizer.num_tokens
        self.num_tokens = n + 1  # + CLS
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, n + 1, embed_dim))  # học được
        self.pos_drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([
            EncoderBlock(embed_dim, num_heads, mlp_ratio, dropout, attn_impl, attn_dropout)
            for _ in range(depth)
        ])
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, num_classes)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.trunc_normal_(self.cls_token, std=0.02)

    def forward(self, x):                                   # (B, 1, 28, 28)
        t = self.tokenizer(x)                               # (B, N, D)
        cls = self.cls_token.expand(t.size(0), -1, -1)      # (B, 1, D)
        t = torch.cat([cls, t], dim=1) + self.pos_embed     # (B, N+1, D)
        t = self.pos_drop(t)
        for blk in self.blocks:
            t = blk(t)
        t = self.norm(t)
        return self.head(t[:, 0])                           # logits từ token CLS


@torch.no_grad()
def copy_weights_torch_to_scratch(scratch: ImageTransformer, ref: ImageTransformer) -> None:
    """Đưa model scratch về đúng trọng số của model torch (cùng kiến trúc).

    Các tham số ngoài MSA có cùng tên ở hai model nên chép thẳng; phần MSA chép bằng
    load_from_torch. Dùng để hai lần chạy (scratch / torch) xuất phát từ CÙNG trọng số,
    khi đó khác biệt kết quả chỉ còn do cách tính MSA.
    """
    ref_sd = ref.state_dict()
    for name, t in scratch.state_dict().items():
        if name in ref_sd and ref_sd[name].shape == t.shape:
            t.copy_(ref_sd[name])
    for s_blk, r_blk in zip(scratch.blocks, ref.blocks, strict=True):
        s_blk.attn.load_from_torch(r_blk.attn.mha)

In [ ]:
# Sau khi chép trọng số, model tự viết và model PyTorch phải cho cùng output (float64)
x64 = torch.randn(3, 1, 28, 28, dtype=torch.float64)
for tok in ["patch4", "row", "cnn"]:
    kw = dict(tokenizer=tok, embed_dim=32, depth=2, num_heads=4)
    ref_model = ImageTransformer(attn_impl="torch", **kw).double().eval()
    my_model = ImageTransformer(attn_impl="scratch", **kw).double().eval()
    copy_weights_torch_to_scratch(my_model, ref_model)
    diff = (ref_model(x64) - my_model(x64)).abs().max().item()
    print(f"{tok:7s} lệch lớn nhất {diff:.1e} | tham số {count_params(my_model):,} vs {count_params(ref_model):,}")
for tok in ["patch4", "patch7", "row", "cnn"]:
    m = ImageTransformer(tok, "scratch", EMBED_DIM, DEPTH, HEADS, MLP_RATIO, DROPOUT)
    print(f"Cấu hình huấn luyện, tokenizer {tok:7s}: {m.num_tokens} token (gồm CLS), {count_params(m):,} tham số")

## 4. Huấn luyện

- Cùng tokenizer: bản **tự viết** được khởi tạo bằng **đúng trọng số** của bản PyTorch (dựng model PyTorch với seed
  cố định rồi chép sang) và dùng cùng thứ tự batch, dropout attention = 0. Khác biệt kết quả chỉ còn đến từ cách tính MSA.
- Giữa các tokenizer: cùng encoder và siêu tham số — AdamW (LR 1e-3, weight decay 0.05), warmup 1 epoch rồi cosine.

In [ ]:
def build_transformer(tokenizer, impl, seed=SEED):
    kw = dict(tokenizer=tokenizer, embed_dim=EMBED_DIM, depth=DEPTH, num_heads=HEADS,
              mlp_ratio=MLP_RATIO, dropout=DROPOUT, attn_dropout=0.0)
    set_seed(seed)
    ref = ImageTransformer(attn_impl="torch", **kw)
    if impl == "torch":
        return ref
    model = ImageTransformer(attn_impl="scratch", **kw)
    copy_weights_torch_to_scratch(model, ref)
    return model


for run in RUNS:
    tokenizer, impl = run.split(":")
    model = build_transformer(tokenizer, impl)
    set_seed(SEED)
    data = get_data(DATA_ROOT, batch_size=BATCH_SIZE, loader_seed=SEED)
    cfg = TrainConfig(epochs=EPOCHS, lr=LR, weight_decay=WEIGHT_DECAY, warmup_epochs=WARMUP_EPOCHS, seed=SEED,
                      extra={"exercise": "E2", "tokenizer": tokenizer, "attn_impl": impl,
                             "num_tokens": model.num_tokens, "embed_dim": EMBED_DIM, "depth": DEPTH,
                             "heads": HEADS, "mlp_ratio": MLP_RATIO, "dropout": DROPOUT, "batch_size": BATCH_SIZE})
    fit(model, data, cfg, device, os.path.join(RESULTS, f"{tokenizer}_{impl}"), name=f"E2-{tokenizer}-{impl}")

## 5. Kết quả

In [ ]:
runs = {}
for run in RUNS:
    tok, impl = run.split(":")
    runs[(tok, impl)] = load_run(os.path.join(RESULTS, f"{tok}_{impl}"))
labels = {k: f"{TOK_LABEL[k[0]]} · {IMPL_LABEL[k[1]]}" for k in runs}
table = pd.DataFrame([{**summary_row(labels[k], r), "Số token": r["summary"]["config"]["extra"]["num_tokens"]}
                      for k, r in runs.items()]).set_index("Mô hình")
display(table)
table.to_csv(os.path.join(FIGURES, "e2_table.csv"))

### MSA tự viết vs PyTorch (cùng tokenizer, cùng khởi tạo)

In [ ]:
rows = []
for tok in TOK_LABEL:
    if (tok, "scratch") in runs and (tok, "torch") in runs:
        a, b = runs[(tok, "scratch")], runs[(tok, "torch")]
        sa, sb = a["summary"], b["summary"]
        rows.append({"Tokenizer": TOK_LABEL[tok],
                     "Test acc tự viết (%)": round(100 * sa["test_acc"], 2),
                     "Test acc PyTorch (%)": round(100 * sb["test_acc"], 2),
                     "Val acc lệch tối đa qua các epoch": round(100 * (a["history"]["val_acc"]
                                                                       - b["history"]["val_acc"]).abs().max(), 2),
                     "Giây/epoch tự viết": round(sa["mean_epoch_time"], 2),
                     "Giây/epoch PyTorch": round(sb["mean_epoch_time"], 2),
                     "Suy luận tự viết (s)": round(sa["test_inference_time"], 3),
                     "Suy luận PyTorch (s)": round(sb["test_inference_time"], 3)})
display(pd.DataFrame(rows).set_index("Tokenizer"))

### Đường cong học
Màu: tokenizer; nét liền: MSA tự viết, nét gạch: PyTorch.

In [ ]:
tok_order = [t for t in TOK_LABEL if any(k[0] == t for k in runs)]
colors = {k: SERIES[tok_order.index(k[0])] for k in runs}
styles = {k: "-" if k[1] == "scratch" else "--" for k in runs}
plot_curves(runs, labels, colors, styles, save_as=os.path.join(FIGURES, "e2_curves.png"))

### Attention của token CLS (khối encoder cuối, patch 4×4, MSA tự viết)

Trung bình theo head, mức chú ý từ token CLS tới 49 patch, phóng về 28×28 và phủ lên ảnh.
Vùng sáng là các patch mô hình dựa vào nhiều nhất để phân loại.

In [ ]:
@torch.no_grad()
def cls_attention_map(model, x):
    model.eval()
    t = model.tokenizer(x)
    cls = model.cls_token.expand(t.size(0), -1, -1)
    t = torch.cat([cls, t], dim=1) + model.pos_embed
    for blk in model.blocks[:-1]:
        t = blk(t)
    last = model.blocks[-1]
    _, attn = last.attn(last.norm1(t), return_attn=True)   # (B, h, N+1, N+1)
    a = attn[:, :, 0, 1:].mean(1)                          # CLS -> các patch, trung bình theo head
    side = model.tokenizer.grid[0]
    return a.reshape(-1, side, side).cpu().numpy()


ckpt = os.path.join(RESULTS, "patch4_scratch", "best.pt")
if os.path.isfile(ckpt):
    vit = build_transformer("patch4", "scratch").to(device)
    vit.load_state_dict(torch.load(ckpt, map_location=device))
    data = get_data(DATA_ROOT, batch_size=BATCH_SIZE, loader_seed=SEED)
    x_test, y_test = next(iter(data.test))
    pick = [int(np.where(y_test.numpy() == c)[0][0]) for c in range(10)]
    maps = cls_attention_map(vit, x_test[pick].to(device))
    with torch.no_grad():
        preds = vit(x_test[pick].to(device)).argmax(1).cpu().numpy()
    fig, axes = plt.subplots(2, 10, figsize=(14, 3.3))
    for j, i in enumerate(pick):
        axes[0, j].imshow(TEST_IMAGES[i], cmap="gray_r")
        axes[0, j].set_title(f"{CLASSES[int(y_test[i])]}\nđoán: {CLASSES[int(preds[j])]}", fontsize=7)
        up = np.kron(maps[j], np.ones((4, 4)))
        axes[1, j].imshow(TEST_IMAGES[i], cmap="gray_r")
        axes[1, j].imshow(up, cmap="inferno", alpha=0.6)
        for ax in axes[:, j]:
            ax.axis("off")
    fig.savefig(os.path.join(FIGURES, "e2_cls_attention.png"))
    plt.show()
else:
    print("Chưa có checkpoint patch4_scratch — cần chạy ô huấn luyện với 'patch4:scratch' trong RUNS.")

## Ghi chú khi viết nhận xét

- Tự viết vs PyTorch: accuracy và đường cong có trùng nhau không; khác biệt thời gian/bộ nhớ nằm ở đâu (xem benchmark theo N).
- Tokenizer: số token ảnh hưởng thế nào đến accuracy và thời gian; vì sao CNN-stem khác patch.
- So với CNN của E1: Transformer có vượt được không, vì sao.

## Lưu kết quả

In [ ]:
# Nén kết quả để tải về (Colab). Chạy trên máy cá nhân thì kết quả đã nằm sẵn trong thư mục results/ và figures/.
import shutil

archive = shutil.make_archive(f"{EXERCISE}_results", "zip", ".", base_dir="results")
shutil.make_archive(f"{EXERCISE}_figures", "zip", ".", base_dir="figures")
print("Đã tạo", archive, "và", f"{EXERCISE}_figures.zip")
try:
    from google.colab import files
    files.download(archive)
    files.download(f"{EXERCISE}_figures.zip")
except ImportError:
    pass